# 01a — Auto-Caption Reference Images

Generates `.txt` sidecar caption files for each reference image using JoyCaption.
These captions are used for SDXL LoRA training in `01b_train_sdxl_lora.ipynb`.

**Runtime:** T4 is fine (captioning is light). A100 optional.

**Prerequisites:** JoyCaption is a gated model — you need a free HuggingFace account.
1. Sign up at https://huggingface.co (free)
2. Accept the model terms at https://huggingface.co/fancyfeast/llama-joycaption-beta-one-hf-llava
3. Create a read token at https://huggingface.co/settings/tokens
4. Paste it in Cell 1b below when prompted

**Alternative:** If you don't want to create an HF account, use the `llava-hf/llava-1.5-7b-hf` 
model instead (fully public, similar caption quality). Change `USE_JOYCAPTION = True` to `False` in Cell 1b.

**Flow:**
1. HuggingFace login
2. Upload reference images to Drive (or Colab tmp)
3. Run captioner on each image
4. Prepend trigger token to each caption
5. Save `.txt` sidecar files next to each image

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ai_character_studio'
CHARACTER_NAME = 'Aria'       # ← change this
TRIGGER_TOKEN  = 'ohwx_aria'  # ← change this (must match what you use in training)

# Captioner choice:
# False = LLaVA 1.5 7B (default — fully public, works out-of-the-box, great for LoRA training)
# True  = JoyCaption Beta One (highest quality, but requires HF account + model access at
#         https://huggingface.co/fancyfeast/llama-joycaption-beta-one-hf-llava)
USE_JOYCAPTION = False

import os
CHAR_DIR    = f'{DRIVE_BASE}/characters/{CHARACTER_NAME}'
REF_DIR     = f'{CHAR_DIR}/reference-images'
CAPTION_DIR = f'{CHAR_DIR}/captions'
os.makedirs(REF_DIR, exist_ok=True)
os.makedirs(CAPTION_DIR, exist_ok=True)

print(f'Character: {CHARACTER_NAME} / trigger: {TRIGGER_TOKEN}')
print(f'Reference images dir: {REF_DIR}')
print(f'Captioner: {"JoyCaption Beta One" if USE_JOYCAPTION else "LLaVA 1.5 7B (public, default)"}')
print('Upload your reference images to the Drive folder above, then run the next cells.')

In [ ]:
# (Optional) Upload images directly from this notebook
from google.colab import files
import shutil

print('Select your reference images to upload...')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = f'{REF_DIR}/{fname}'
    with open(dest, 'wb') as f:
        f.write(data)
    print(f'  Saved {fname} → {dest}')

In [ ]:
!pip install -q transformers torch Pillow huggingface_hub

from transformers import AutoProcessor, LlavaForConditionalGeneration
import torch, os

if USE_JOYCAPTION:
    # JoyCaption Beta One — gated model. Two ways to authenticate:
    # Option 1 (recommended): Add HF_TOKEN to Colab Secrets (key icon in left sidebar → "Add secret")
    # Option 2: paste token directly below (less secure — don't commit this)
    from google.colab import userdata
    try:
        hf_token = userdata.get('HF_TOKEN')
        print('HF_TOKEN loaded from Colab Secrets.')
    except Exception:
        # Fallback: paste token here
        from getpass import getpass
        hf_token = getpass('Paste your HuggingFace read token (hidden): ')

    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    MODEL_ID = 'fancyfeast/llama-joycaption-beta-one-hf-llava'
else:
    MODEL_ID = 'llava-hf/llava-1.5-7b-hf'

print(f'Loading captioner: {MODEL_ID} ...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16
).to('cuda' if torch.cuda.is_available() else 'cpu')
print('Captioner loaded.')

In [ ]:
from PIL import Image
from pathlib import Path
import torch

CAPTION_INSTRUCTION = (
    'Describe this image in detail for use as a training caption for a diffusion model. '
    'Focus on: the pose and body position, facial expression, clothing and accessories, '
    'lighting and background. Do NOT describe the character identity or face structure — '
    'that will be represented by a trigger token. Be specific and concrete. '
    'Keep the caption under 80 words.'
)

def caption_image(image_path: str) -> str:
    image = Image.open(image_path).convert('RGB')
    # Confirmed working format from diagnostic: list content → 576 image tokens
    conversation = [{
        'role': 'user',
        'content': [{'type': 'image'}, {'type': 'text', 'text': CAPTION_INSTRUCTION}]
    }]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    inputs = processor(text=prompt, images=[image], return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generated = processor.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated.strip()

exts = {'.jpg', '.jpeg', '.png', '.webp'}
images = [p for p in Path(REF_DIR).iterdir() if p.suffix.lower() in exts]
print(f'Found {len(images)} reference images. Starting captioning...')

captions = {}
for img_path in sorted(images):
    print(f'Captioning {img_path.name} ...')
    try:
        raw_caption = caption_image(str(img_path))
        full_caption = f'{TRIGGER_TOKEN}, {raw_caption}'
    except Exception as e:
        print(f'  ERROR: {e} — skipping')
        continue
    captions[img_path.name] = full_caption
    Path(REF_DIR, img_path.stem + '.txt').write_text(full_caption)
    Path(CAPTION_DIR, img_path.stem + '.txt').write_text(full_caption)
    print(f'  → {full_caption[:100]}...')

print(f'\n✅ {len(captions)}/{len(images)} captions saved.')

In [ ]:
from IPython.display import display, HTML
from pathlib import Path
import base64

def img_to_b64(path, max_size=200):
    from PIL import Image
    img = Image.open(path).convert('RGB')
    img.thumbnail((max_size, max_size))
    from io import BytesIO
    buf = BytesIO()
    img.save(buf, format='JPEG', quality=80)
    return base64.b64encode(buf.getvalue()).decode()

rows = []
for fname, caption in sorted(captions.items()):
    img_path = Path(REF_DIR) / fname
    try:
        b64 = img_to_b64(str(img_path))
        img_tag = f'<img src="data:image/jpeg;base64,{b64}" style="width:180px;height:180px;object-fit:cover;border-radius:4px;">'
    except Exception:
        img_tag = '<span style="color:#888">image not found</span>'

    # Highlight the trigger token in the caption
    highlighted = caption.replace(
        TRIGGER_TOKEN,
        f'<span style="background:#1a3a1a;color:#7fff7f;padding:1px 4px;border-radius:3px;font-weight:bold;">{TRIGGER_TOKEN}</span>',
        1
    )
    rows.append(f'''
      <tr>
        <td style="padding:8px;vertical-align:top;width:200px;">{img_tag}</td>
        <td style="padding:8px;vertical-align:top;font-family:monospace;font-size:12px;
                   background:#1e1e1e;color:#d4d4d4;white-space:pre-wrap;line-height:1.5;">{highlighted}</td>
      </tr>
    ''')

html = f'''
<style>
  .caption-table {{ border-collapse: collapse; width: 100%; background: #141414; }}
  .caption-table tr {{ border-bottom: 1px solid #333; }}
  .caption-table tr:hover td {{ background: #1a1a2e !important; }}
</style>
<h3 style="color:#ccc;font-family:sans-serif;">Caption Review — {len(captions)} images
  <span style="font-size:12px;color:#888;">(edit .txt files in Drive if any caption needs fixing)</span>
</h3>
<table class="caption-table">
  {''.join(rows)}
</table>
'''
display(HTML(html))

In [ ]:
# Register character in library (optional — also done in training notebook)
import sys, json
# If running from Colab, library.py isn't installed — use inline version
metadata = {
    'name': CHARACTER_NAME,
    'trigger': TRIGGER_TOKEN,
    'base_model': 'sdxl',
    'ref_count': len(captions),
}
meta_path = f'{CHAR_DIR}/metadata.json'
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f'metadata.json written: {meta_path}')
print('\n✅ Done. Run 01b_train_sdxl_lora.ipynb next to train the character LoRA.')